# colony4

> Phase 18, Stage 4 (`specs/multiagent/phase18/stage4.md`, FR-A24): guards that scale. Three board options on top of assigned guards: guard copies (`guardDeal`), failure flags (`failFlag`) and squads of 8 (`squads`), plus two combined arms, at 16 and 64 agents with independent liars (16 agents also clean), Stage 1's seeds. `ARMS4`, `stage4Grid` (80 cells), `stage4PilotGrid`, `reusedCells4` (56 references replayed from the cache), `stage4Estimate`, `costRule4`, `pilotCheck4` and `STAGE4_PLAN`.

In [ ]:
#| default_exp colony4

In [ ]:
#| hide
from nbdev.showdoc import *

## Arms, grids and plan

In [ ]:
#| export
import numpy as np

from explRL.clue_colony import checkOptions
from explRL.colony64 import stage64Grid
from explRL.colony_grid import COLONY_ARMS, COLONY_SEEDS, PILOT_SEED, colonyCell, stage1Grid
from explRL.fix_grid import cellCalls

_G = dict(colony=True, guards='quarter')
ARMS4 = {**COLONY_ARMS,                                  # COLONY_ARMS itself unchanged (COLONY_PLAN freezes it)
         'guardDeal': dict(_G, guardDeal=True), 'failFlag': dict(_G, failFlag=True), 'squads': dict(_G, squads=8),
         'guardDealFlag': dict(_G, guardDeal=True, failFlag=True),
         'squadsAll': dict(_G, guardDeal=True, failFlag=True, squads=8)}
STAGE4_CELLS = {16: [('guardDeal', 'clean'), ('guardDeal', 'independent'), ('failFlag', 'clean'), ('failFlag', 'independent'),
                     ('squads', 'independent'), ('guardDealFlag', 'independent')],
                64: [('guardDeal', 'independent'), ('failFlag', 'independent'), ('squads', 'independent'),
                     ('squadsAll', 'independent')]}
REUSED4 = {8: [('colonyGuards', 'independent')],
           16: [('colonyGuards', 'clean'), ('colonyGuards', 'independent'), ('plain', 'clean'), ('plain', 'independent')],
           64: [('colonyGuards', 'independent'), ('plain', 'independent')]}
PILOT4 = [('guardDeal', 64), ('squads', 64), ('failFlag', 16)]


def _cell(study, arm, N, seed, fault):
    c = colonyCell(study, arm, N, seed, fault, arms=ARMS4)
    checkOptions(c['cfg']['world'], N, c['cfg']['nLiar'])
    return c


def stage4Grid(seeds=COLONY_SEEDS):
    "Study colony4: STAGE4_CELLS x seeds (80 cells; 87,040 calls)."
    return [_cell('colony4', a, N, s, f) for N, cs in STAGE4_CELLS.items() for a, f in cs for s in seeds]


def stage4PilotGrid():
    "The pilot (not analysed): guardDeal and squads at N 64, failFlag at N 16, independent liars, seed 1890."
    return [_cell('colony4Pilot', a, N, PILOT_SEED, 'independent') for a, N in PILOT4]


def reusedCells4(seeds=COLONY_SEEDS):
    "The 56 reference cells: Stage 1's (study colony1) at 8 and 16 agents and the 64-agent run's (colony64)."
    keep = lambda c: (c['level'].split('|')[0], c['level'].split('|')[1]) in REUSED4.get(c['N'], [])
    return [c for c in stage1Grid(seeds) + stage64Grid(seeds) if keep(c)]


STAGE4_PLAN = dict(
    version=1, frozen=True, plan='specs/multiagent/phase18/stage4.md',
    arms={a: ARMS4[a] for a in ('guardDeal', 'failFlag', 'squads', 'guardDealFlag', 'squadsAll')},
    cells={N: [f'{a}|{f}' for a, f in cs] for N, cs in STAGE4_CELLS.items()}, seeds=COLONY_SEEDS,
    reused={N: [f'{a}|{f}' for a, f in cs] for N, cs in REUSED4.items()},
    primary=dict(H18_13='guardDeal', H18_14='failFlag', H18_15='squads',
                 test='arm - colonyGuards, honestPooled, independent liars, N 64, 8 seeds paired: one-sided exact sign-flip > 0; '
                      'Holm over the three', alpha=0.05),
    robustness='each primary repeated on welfareHonest; an estimate of the opposite sign marks the primary not robust',
    secondary=dict(
        did='per arm run at both sizes (guardDeal, failFlag, squads): per seed (arm - colonyGuards)@64 - (arm - colonyGuards)@16; '
            'mean, bootstrap interval, two-sided sign-flip p; colonyGuards loses 0.035 of advantage over plain from 16 to 64',
        advantage='each arm minus plain at 16 and 64 agents', squads8='squads@64 - colonyGuards@8, same seeds (8-agent level 0.284)',
        cleanCost='guardDeal and failFlag clean - colonyGuards clean at 16', combined='guardDealFlag@16, squadsAll@64 against colonyGuards',
        mechanisms=dict(guardDeal='guardHeldLies >= 0.99', failFlag='usesPerUser lower than colonyGuards; flagsOnTrue <= 0.20',
                        squads='openersPerLie and inTokPerCall near the 8-agent values (1.28 openers, 840 tokens)')),
    pilot=dict(cells=[f'{a}|independent N {N}' for a, N in PILOT4], seed=PILOT_SEED, maxInvalid=0.05, minGuardHeld=0.99,
               minContestGuarded=0.60, maxFlagsOnTrue=0.20, maxSquadTokens=1.5,
               fallback='guards overloaded (contest rate under 0.60): propose the triage sentence to the user'),
    costRule=dict(run=60.0, otherwise='stop and ask', projection="Stage 4's calls x the pilot's cost per call by board kind"),
    forecast=dict(file='specs/multiagent/phase18/colony4_predictions.json', commit='1e9c53d',
                  sha256='49de27cb564fda7f91d800bb7a76ba449ba9a545fa162436cac97bc54ac92a2b'), cap=250.0,
    nboot=2000, exploratory=True)

## Estimate and gates

In [ ]:
#| export
def boardKind(c):
    "'squads' for squad boards, else the team size: the pilot's cost per call is measured per kind."
    return 'squads' if c['cfg']['world'].get('squads') else c['N']


def stage4Estimate(cells, perCall):
    "Calls and cost per board kind, from the pilot's cost per call {16, 64, 'squads': dollars}."
    calls = {}
    for c in cells:
        calls[boardKind(c)] = calls.get(boardKind(c), 0) + cellCalls(c)
    return dict(calls=sum(calls.values()), byKind=calls, cost=float(sum(n * perCall[k] for k, n in calls.items())))


def costRule4(projected, plan=STAGE4_PLAN):
    "Run if Stage 4 projects at most plan['costRule']['run'] dollars, else stop and ask."
    return dict(verdict='run' if projected <= plan['costRule']['run'] else 'stop', projected=float(projected))


def ref8Tokens(rows1):
    "Mean input tokens per call of Stage 1's 8-agent colonyGuards boards with independent liars."
    v = [float(r['inputTokens']) / float(r['calls']) for r in rows1
         if r['level'] == 'colonyGuards|independent' and int(float(r['N'])) == 8 and float(r['calls'])]
    return float(np.mean(v)) if v else float('nan')


def pilotCheck4(rows, ref8, plan=STAGE4_PLAN):
    "The pilot's gates (FR-A24) and the cost rule's verdict -> dict(ok, reasons, values, perCall, verdict, projected)."
    P, f = plan['pilot'], lambda r, k: float(r[k])
    by = {(r['level'].split('|')[0], int(float(r['N']))): r for r in rows}
    calls = sum(f(r, 'calls') for r in rows) or 1.0
    vals = dict(invalid=sum(f(r, 'invalid') for r in rows) / calls,
                guardHeld=f(by[('guardDeal', 64)], 'guardHeldLies'), contestGuarded=f(by[('guardDeal', 64)], 'contestRateGuarded'),
                flagsOnTrue=f(by[('failFlag', 16)], 'flagsOnTrue'), squadTokens=f(by[('squads', 64)], 'inTokPerCall'), ref8=ref8)
    reasons = [m for m, bad in (
        (f"invalid {vals['invalid']:.3f} >= {P['maxInvalid']}", vals['invalid'] >= P['maxInvalid']),
        (f"guard-held lies {vals['guardHeld']:.3f} < {P['minGuardHeld']}", not vals['guardHeld'] >= P['minGuardHeld']),
        (f"guard contest rate {vals['contestGuarded']:.3f} < {P['minContestGuarded']} ({P['fallback']})",
         not vals['contestGuarded'] >= P['minContestGuarded']),
        (f"flags on true values {vals['flagsOnTrue']:.3f} > {P['maxFlagsOnTrue']} (or no failure recorded)",
         not vals['flagsOnTrue'] <= P['maxFlagsOnTrue']),
        (f"squad prompt {vals['squadTokens']:.0f} tokens > {P['maxSquadTokens']} x {ref8:.0f}",
         not vals['squadTokens'] <= P['maxSquadTokens'] * ref8)) if bad]
    perCall = {k: f(by[(a, N)], 'cost') / f(by[(a, N)], 'calls') for k, (a, N) in ((64, ('guardDeal', 64)),
                                                                                   ('squads', ('squads', 64)), (16, ('failFlag', 16)))}
    rule = costRule4(stage4Estimate(stage4Grid(), perCall)['cost'], plan)
    return dict(ok=not reasons and rule['verdict'] == 'run', reasons=reasons, values=vals, perCall=perCall, **rule)

## Tests

In [ ]:
from collections import Counter

g = stage4Grid()
assert len(g) == 80 and sum(cellCalls(c) for c in g) == 87040
assert Counter(c['N'] for c in g) == {16: 48, 64: 32}
w64 = {c['level']: c['cfg']['world'] for c in g if c['N'] == 64}
assert w64['squadsAll|independent'] == dict(w64['squadsAll|independent'], guardDeal=True, failFlag=True, squads=8, guards=16)
assert all(c['cfg']['nLiar'] == (c['N'] // 4 if 'independent' in c['level'] else 0) for c in g)
assert [(c['level'], c['N']) for c in stage4PilotGrid()] == [('guardDeal|independent', 64), ('squads|independent', 64), ('failFlag|independent', 16)]
r = reusedCells4()
assert len(r) == 56 and Counter((c['cfg']['study'] if 'study' in c['cfg'] else c['study'], c['N']) for c in r) == {('colony1', 8): 8, ('colony1', 16): 32, ('colony64', 64): 16}
assert 'squads' not in COLONY_ARMS and 'guardDeal' not in COLONY_ARMS
e = stage4Estimate(g, {16: 0.0004, 64: 0.0009, 'squads': 0.0003})
assert e['byKind'] == {16: 21760, 'squads': 34560, 64: 30720} and abs(e['cost'] - (21760 * 0.0004 + 34560 * 0.0003 + 30720 * 0.0009)) < 1e-9
assert costRule4(59.0)['verdict'] == 'run' and costRule4(61.0)['verdict'] == 'stop'

In [ ]:
# the pilot's gates both ways (synthetic rows)
def prow(arm, N, **kw):
    base = dict(level=f'{arm}|independent', N=N, calls=1000, invalid=10, cost=0.5, guardHeldLies=1.0, contestRateGuarded=0.7,
                flagsOnTrue=0.05, inTokPerCall=900)
    return {**base, **kw}
good = [prow('guardDeal', 64), prow('squads', 64, cost=0.3), prow('failFlag', 16, cost=0.4)]
pc = pilotCheck4(good, 840)
assert pc['ok'] and not pc['reasons'] and pc['verdict'] == 'run', pc
bad = [prow('guardDeal', 64, contestRateGuarded=0.5), prow('squads', 64, inTokPerCall=2000), prow('failFlag', 16, flagsOnTrue=0.3, invalid=200)]
pc = pilotCheck4(bad, 840)
assert not pc['ok'] and len(pc['reasons']) == 4, pc['reasons']
dear = [prow('guardDeal', 64, cost=5.0), prow('squads', 64), prow('failFlag', 16)]
assert pilotCheck4(dear, 840)['verdict'] == 'stop'
none = [prow('guardDeal', 64), prow('squads', 64), prow('failFlag', 16, flagsOnTrue=float('nan'))]
assert not pilotCheck4(none, 840)['ok']                              # no failure recorded: the flags were never tested
assert abs(ref8Tokens([dict(level='colonyGuards|independent', N='8', inputTokens='8400', calls='10')]) - 840) < 1e-9